# pydantic to structure gemini output

In [ ]:
from dotenv import load_dotenv
import os
from google import genai


# load_dotenv()

client = genai.Client() # api_key=os.getenv("GEMINI_API_KEY"))

# response = client.models.generate_content(
#     model="gemini-2.5-flash", contents="tell a gbg joke"
# )
# # response.text
# print(response.text)

In [ ]:
def ask_llm(prompt):

    response = client.models.generate_content(
        model="gemini-2.5-flash", 
        contents=prompt
    )

    return response.text

In [ ]:
ask_llm("tell me a secret")

# try to get data

In [ ]:
response = ask_llm("""
    Du är en expert inom köp och sälj av bostäder, likt en proffsig mäklare.
    Generera bostadspriser, månadsavgifter, address, stad, boarea i jsonformat (ej markdown)

    Exempel:
            {
                "address": "Fågelvägen 5,
                "price_sek": 3000000,
                "city": "Göteborg",
                "monthly_fee": 4000,
                "area": 60
            }   
                   
    Ge mig en lista på 5 bostäder
""")

response

In [ ]:
print(response)

## parse and validate data

In [ ]:
from pydantic import BaseModel, Field
import json 

class Apartment(BaseModel):
    address: str 
    city: str 
    price_sek: int = Field(gt=1000000, lt = 8000000) 
    monthly_fee: int 
    area: int 

class ApartmentList(BaseModel):
    objects: list[Apartment]


apartments = ApartmentList.model_validate({"objects": json.loads(response)})
apartments

In [ ]:
apartments.objects

In [ ]:
apartments.objects[1].address, apartments.objects[1].city

In [ ]:
addresses = [apartment.address for apartment in apartments.objects ]

addresses

In [ ]:
addresses = [
    apartment.address
    for apartment in apartments.objects
    if apartment.price_sek < 4000000
]

addresses

## filter out adress, city, price, monthly_fee for the interval 4m-8m

In [ ]:
import pandas as pd



apartments_price_range = [
    [apartment.address, apartment.city, apartment.price_sek, apartment.monthly_fee]
    for apartment
    in apartments.objects
    if 4000000 < apartment.price_sek < 8000000
    # if home.price_sek > 4000000 and home.price_sek < 8000000
]

apartments_price_range



# price_limited_apartments

## convert to df

In [ ]:
# Filter
filtered_apartments = [
    apartment for apartment in apartments.objects if 4_000_000 < apartment.price_sek < 8_000_000
]

filtered_apartments

In [ ]:
df_filtered = pd.DataFrame(
    [
        apartments.model_dump(include={"address", "city", "price_sek", "monthly_fee"})
        for apartments in filtered_apartments
    ]
)
df_filtered

In [ ]:
df_filtered.to_csv("filtered_apartments.csv", index=False)

### import to duckbd

In [ ]:
import duckdb

db_table = duckdb.read_csv("filtered_apartments.csv")
print(db_table)


In [ ]:
# 1. Anslut till din DuckDB-databas (skapar filen om den inte finns)
con = duckdb.connect("apartments.duckdb")

# 2. Skapa en tabell från din CSV
con.execute("""
    CREATE OR REPLACE TABLE apartments AS
    SELECT * FROM read_csv_auto('filtered_apartments.csv')
""")

# 3. Starta DuckDB:s UI (öppnas i webbläsaren på http://localhost:4213)
con.execute("CALL start_ui();")